# Assignment 01: House Price Prediction System

## 1. System and Problem Definition

**Hệ thống: Dự đoán Giá Nhà**

- **Vấn đề:** Dự đoán giá bán của bất động sản dựa trên các đặc điểm.
- **Đầu vào:** `x` = [Area, Floors, Bedrooms, Bathrooms, Legal_encoded, Furniture_encoded]
- **Đầu ra:** `y` ∈ R (Price in billions VND)
- **Loại bài toán:** Hồi quy (Regression)

**Sơ đồ hệ thống:**
```
Đặc điểm nhà → [Tiền xử lý & Mã hóa & Chuẩn hóa] → Vector đặc trưng → [ML Model] → Giá dự đoán
```

## 2. Intelligent System Diagram

```
┌──────────────────────────────────────────────────────────────────────┐
│                    HỆ THỐNG DỰ ĐOÁN GIÁ NHÀ                          │
└──────────────────────────────────────────────────────────────────────┘
                                 │
                                 ▼
┌───────────────────────────────────────────────────────────────────────┐
│  INPUT: Đặc điểm bất động sản                                         │
│  - Area (Diện tích, m²)                                               │
│  - Floors (Số tầng)                                                   │
│  - Bedrooms (Số phòng ngủ)                                            │
│  - Bathrooms (Số phòng tắm)                                           │
│  - Legal status (Tình trạng pháp lý)                                  │
│  - Furniture state (Tình trạng nội thất)                              │
└───────────────────────────────────────────────────────────────────────┘
                                 │
                                 ▼
┌───────────────────────────────────────────────────────────────────────┐
│  REPRESENTATION: Feature Vector x ∈ R⁶                                │
│  - Numerical: Area, Floors, Bedrooms, Bathrooms                       │
│  - Categorical encoded: Legal_encoded (0/1), Furniture_encoded (0/1/2)│
│  - Điền giá trị thiếu bằng median (số) và most_frequent (phân loại)   │
│  - Chuẩn hóa (StandardScaler): mean=0, std=1                          │
└───────────────────────────────────────────────────────────────────────┘
                                 │
                                 ▼
┌───────────────────────────────────────────────────────────────────────┐
│  MODEL: Random Forest (hoặc Linear Regression, Decision Tree, SVR,    │
│          Gradient Boosting)                                           │
│  - Học mối quan hệ từ dữ liệu huấn luyện                              │
│  - Dự đoán giá cho bất động sản mới                                   │
└───────────────────────────────────────────────────────────────────────┘
                                 │
                                 ▼
┌───────────────────────────────────────────────────────────────────────┐
│  OUTPUT:                                                              │
│  - Giá dự đoán (tỷ VND)                                               │
└───────────────────────────────────────────────────────────────────────┘
```

## 3. Dataset Source

**Nguồn dữ liệu:** Cung cấp trong assignment
- File: `vietnam_housing_dataset.csv`
- Link: https://www.kaggle.com/datasets/nguyentiennhan/vietnam-housing-dataset-2024

## 4. Dataset Description

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.base import BaseEstimator, TransformerMixin
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

In [ ]:
# Đọc dữ liệu
housing_df = pd.read_csv(r"C:\DATA\vietnam_housing_dataset.csv")
print("="*60)
print("THÔNG TIN DỮ LIỆU GIÁ NHÀ")
print("="*60)
print(f"Số lượng mẫu: {housing_df.shape[0]}")
print(f"Số lượng cột: {housing_df.shape[1]}")

In [ ]:
# Chọn cột quan trọng
selected_cols = ['Area', 'Floors', 'Bedrooms', 'Bathrooms', 'Legal status', 'Furniture state', 'Price']
housing_selected = housing_df[selected_cols].copy()
housing_selected.dropna(subset=['Price'], inplace=True)

print(f"\nSau khi chọn cột và loại bỏ dòng thiếu Price:")
print(f"Số lượng mẫu: {housing_selected.shape[0]}")
print(f"Số lượng đặc trưng: {housing_selected.shape[1] - 1}")

In [ ]:
print("\n5 mẫu đầu tiên:")
display(housing_selected.head())

In [ ]:
print("\nTHÔNG TIN CÁC CỘT:")
print("-"*40)
print(housing_selected.dtypes)

In [ ]:
print("\nTHỐNG KÊ MÔ TẢ:")
display(housing_selected.describe())

In [ ]:
print("\nPHÂN PHỐI BIẾN PHÂN LOẠI:")
print("-"*40)
print("\nLegal status:")
print(housing_selected['Legal status'].value_counts())
print("\nFurniture state:")
print(housing_selected['Furniture state'].value_counts())

## 5. Data Representation

**Biểu diễn dữ liệu:**

In [ ]:
# Tạo bảng biểu diễn đặc trưng
representation_df = pd.DataFrame({
    'Đặc trưng': ['Area', 'Floors', 'Bedrooms', 'Bathrooms', 
                  'Legal_encoded', 'Furniture_encoded', 'Price'],
    'Kiểu dữ liệu': ['Numerical', 'Numerical', 'Numerical', 'Numerical', 
                     'Categorical', 'Categorical', 'Numerical'],
    'Biểu diễn': ['float', 'float', 'float', 'float', 'int', 'int', 'float'],
    'Ý nghĩa': ['Diện tích (m²)', 'Số tầng', 'Số phòng ngủ', 'Số phòng tắm',
                'Có giấy tờ: 1, Khác: 0', 'Full:2, Basic:1, Khác:0', 'Giá bán (tỷ VND)']
})
display(representation_df)

## 6. Feature and Target Analysis

In [ ]:
print("="*60)
print("PHÂN TÍCH ĐẶC TRƯNG VÀ MỤC TIÊU")
print("="*60)

In [ ]:
# 3 biểu đồ phân phối
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Biểu đồ 1: Phân phối Area
sns.histplot(housing_selected['Area'].dropna(), bins=30, kde=True, ax=axes[0], color='blue')
axes[0].set_title('Phân phối Diện tích (Area)', fontsize=14)
axes[0].set_xlabel('Area (m²)')
axes[0].set_ylabel('Tần suất')

# Biểu đồ 2: Phân phối Price
sns.histplot(housing_selected['Price'].dropna(), bins=30, kde=True, ax=axes[1], color='green')
axes[1].set_title('Phân phối Giá nhà (Price)', fontsize=14)
axes[1].set_xlabel('Price (tỷ VND)')
axes[1].set_ylabel('Tần suất')

# Biểu đồ 3: Phân phối Floors
sns.histplot(housing_selected['Floors'].dropna(), bins=15, kde=True, ax=axes[2], color='red')
axes[2].set_title('Phân phối Số tầng (Floors)', fontsize=14)
axes[2].set_xlabel('Floors')
axes[2].set_ylabel('Tần suất')

plt.tight_layout()
plt.show()

## 7. Exploratory Data Analysis (EDA)

In [ ]:
print("="*60)
print("PHÂN TÍCH KHÁM PHÁ DỮ LIỆU (EDA)")
print("="*60)

In [ ]:
# 3 biểu đồ phân phối EDA
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Biểu đồ 1: Price theo Legal status
sns.boxplot(x='Legal status', y='Price', data=housing_selected, ax=axes[0])
axes[0].set_title('Giá nhà theo Tình trạng pháp lý', fontsize=14)
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=45)

# Biểu đồ 2: Price theo Furniture state
sns.boxplot(x='Furniture state', y='Price', data=housing_selected, ax=axes[1])
axes[1].set_title('Giá nhà theo Tình trạng nội thất', fontsize=14)
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=45)

# Biểu đồ 3: Scatter Area vs Price
sns.scatterplot(x='Area', y='Price', data=housing_selected, ax=axes[2], alpha=0.5)
axes[2].set_title('Area vs Price', fontsize=14)
axes[2].set_xlabel('Area (m²)')
axes[2].set_ylabel('Price (tỷ VND)')

plt.tight_layout()
plt.show()

In [ ]:
# Ma trận tương quan
numerical_cols = ['Area', 'Floors', 'Bedrooms', 'Bathrooms', 'Price']
corr_matrix = housing_selected[numerical_cols].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f', square=True)
plt.title('Ma trận tương quan các đặc trưng số', fontsize=16)
plt.show()

## 8. Train/Test Split

In [ ]:
# Custom transformer cho mã hóa biến phân loại
class CategoricalEncoder(BaseEstimator, TransformerMixin):
    def __init__(self):
        pass
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        X_encoded = pd.DataFrame()
        if 'Legal status' in X.columns:
            X_encoded['Legal_encoded'] = X['Legal status'].apply(lambda x: 1 if x == 'Have certificate' else 0)
        if 'Furniture state' in X.columns:
            X_encoded['Furniture_encoded'] = X['Furniture state'].apply(
                lambda x: 2 if x == 'Full' else 1 if x == 'Basic' else 0)
        return X_encoded

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import FunctionTransformer

# Chuẩn bị dữ liệu
X_housing = housing_selected.drop('Price', axis=1)
y_housing = housing_selected['Price']

# Định nghĩa các cột
num_cols = ['Area', 'Floors', 'Bedrooms', 'Bathrooms']
cat_cols = ['Legal status', 'Furniture state']

# Pipeline cho biến số
num_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Hàm mã hóa biến phân loại
def encode_categorical(X):
    X = pd.DataFrame(X, columns=cat_cols)

    legal_encoded = X['Legal status'].apply(
        lambda x: 1 if x == 'Have certificate' else 0
    )

    furniture_encoded = X['Furniture state'].apply(
        lambda x: 2 if x == 'Full'
        else 1 if x == 'Basic'
        else 0
    )

    return np.column_stack([
        legal_encoded,
        furniture_encoded
    ])

# Transformer cho biến phân loại
cat_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', FunctionTransformer(encode_categorical))
])

# ColumnTransformer
preprocessor = ColumnTransformer([
    ('num', num_transformer, num_cols),
    ('cat', cat_transformer, cat_cols)
])

# Áp dụng tiền xử lý
X_processed = preprocessor.fit_transform(X_housing)

# Chuyển thành DataFrame
X_processed = pd.DataFrame(
    X_processed,
    columns=[
        'Area',
        'Floors',
        'Bedrooms',
        'Bathrooms',
        'Legal_encoded',
        'Furniture_encoded'
    ],
    index=X_housing.index
)

display(X_processed.head())


In [ ]:
# Chia train/test
X_train, X_test, y_train, y_test = train_test_split(
    X_processed, y_housing, test_size=0.20, random_state=42
)

print("="*60)
print("CHIA DỮ LIỆU TRAIN/TEST")
print("="*60)
print(f"Train set: {X_train.shape[0]} mẫu")
print(f"Test set: {X_test.shape[0]} mẫu")

# Lưu scaler cho ứng dụng
housing_scaler = preprocessor.named_transformers_['num'].named_steps['scaler']
housing_imputer = preprocessor.named_transformers_['num'].named_steps['imputer']

## 9. Baseline

In [ ]:
# Baseline: DummyRegressor (mean)
baseline = DummyRegressor(strategy='mean')
baseline.fit(X_train, y_train)
y_pred_baseline = baseline.predict(X_test)

baseline_mae = mean_absolute_error(y_test, y_pred_baseline)
baseline_r2 = r2_score(y_test, y_pred_baseline)

print("="*60)
print("BASELINE MODEL")
print("="*60)
print(f"Chiến lược: Dự đoán giá trị trung bình")
print(f"MAE: {baseline_mae:.4f}")
print(f"R²: {baseline_r2:.4f}")

## 10. Model 1: Linear Regression

In [ ]:
# Model 1: Linear Regression
print("="*60)
print("MODEL 1: LINEAR REGRESSION")
print("="*60)

lr_model = LinearRegression()
lr_model.fit(X_train, y_train)
y_pred_lr = lr_model.predict(X_test)

mae_lr = mean_absolute_error(y_test, y_pred_lr)
mse_lr = mean_squared_error(y_test, y_pred_lr)
rmse_lr = np.sqrt(mse_lr)
r2_lr = r2_score(y_test, y_pred_lr)

print(f"MAE: {mae_lr:.4f}")
print(f"MSE: {mse_lr:.4f}")
print(f"RMSE: {rmse_lr:.4f}")
print(f"R²: {r2_lr:.4f}")

## 11. Model 2: Decision Tree Regressor

In [ ]:
# Model 2: Decision Tree
print("="*60)
print("MODEL 2: DECISION TREE REGRESSOR")
print("="*60)

dt_model = DecisionTreeRegressor(random_state=42, max_depth=5)
dt_model.fit(X_train, y_train)
y_pred_dt = dt_model.predict(X_test)

mae_dt = mean_absolute_error(y_test, y_pred_dt)
mse_dt = mean_squared_error(y_test, y_pred_dt)
rmse_dt = np.sqrt(mse_dt)
r2_dt = r2_score(y_test, y_pred_dt)

print(f"MAE: {mae_dt:.4f}")
print(f"MSE: {mse_dt:.4f}")
print(f"RMSE: {rmse_dt:.4f}")
print(f"R²: {r2_dt:.4f}")

## 12. Model 3: Random Forest Regressor

In [ ]:
# Model 3: Random Forest
print("="*60)
print("MODEL 3: RANDOM FOREST REGRESSOR")
print("="*60)

rf_model = RandomForestRegressor(random_state=42, n_estimators=100)
rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)

mae_rf = mean_absolute_error(y_test, y_pred_rf)
mse_rf = mean_squared_error(y_test, y_pred_rf)
rmse_rf = np.sqrt(mse_rf)
r2_rf = r2_score(y_test, y_pred_rf)

print(f"MAE: {mae_rf:.4f}")
print(f"MSE: {mse_rf:.4f}")
print(f"RMSE: {rmse_rf:.4f}")
print(f"R²: {r2_rf:.4f}")

## 13. Model 4: Support Vector Regression (SVR)

In [ ]:
# Model 4: SVR
print("="*60)
print("MODEL 4: SUPPORT VECTOR REGRESSION")
print("="*60)

svr_model = SVR(kernel='rbf', C=1.0, epsilon=0.2)
svr_model.fit(X_train, y_train)
y_pred_svr = svr_model.predict(X_test)

mae_svr = mean_absolute_error(y_test, y_pred_svr)
mse_svr = mean_squared_error(y_test, y_pred_svr)
rmse_svr = np.sqrt(mse_svr)
r2_svr = r2_score(y_test, y_pred_svr)

print(f"MAE: {mae_svr:.4f}")
print(f"MSE: {mse_svr:.4f}")
print(f"RMSE: {rmse_svr:.4f}")
print(f"R²: {r2_svr:.4f}")

## 13.5 Model 5: Gradient Boosting Regressor

In [ ]:
# Model 5: Gradient Boosting
print("="*60)
print("MODEL 5: GRADIENT BOOSTING REGRESSOR")
print("="*60)

gb_model = GradientBoostingRegressor(random_state=42, n_estimators=100, learning_rate=0.1)
gb_model.fit(X_train, y_train)
y_pred_gb = gb_model.predict(X_test)

mae_gb = mean_absolute_error(y_test, y_pred_gb)
mse_gb = mean_squared_error(y_test, y_pred_gb)
rmse_gb = np.sqrt(mse_gb)
r2_gb = r2_score(y_test, y_pred_gb)

print(f"MAE: {mae_gb:.4f}")
print(f"MSE: {mse_gb:.4f}")
print(f"RMSE: {rmse_gb:.4f}")
print(f"R²: {r2_gb:.4f}")

## 14. Evaluation

In [ ]:
# Tổng hợp kết quả
results_housing = {
    'Baseline': {'MAE': baseline_mae, 'MSE': mean_squared_error(y_test, y_pred_baseline),
                 'RMSE': np.sqrt(mean_squared_error(y_test, y_pred_baseline)), 'R2': baseline_r2},
    'Linear Regression': {'MAE': mae_lr, 'MSE': mse_lr, 'RMSE': rmse_lr, 'R2': r2_lr},
    'Decision Tree': {'MAE': mae_dt, 'MSE': mse_dt, 'RMSE': rmse_dt, 'R2': r2_dt},
    'Random Forest': {'MAE': mae_rf, 'MSE': mse_rf, 'RMSE': rmse_rf, 'R2': r2_rf},
    'SVR': {'MAE': mae_svr, 'MSE': mse_svr, 'RMSE': rmse_svr, 'R2': r2_svr},
    'Gradient Boosting': {'MAE': mae_gb, 'MSE': mse_gb, 'RMSE': rmse_gb, 'R2': r2_gb}
}

df_results = pd.DataFrame(results_housing).T
print("="*60)
print("TỔNG HỢP KẾT QUẢ CÁC MÔ HÌNH")
print("="*60)
display(df_results.round(4))

In [ ]:
# Scatter plot: Predicted vs Actual (Random Forest)
plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_pred_rf, alpha=0.5)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
plt.xlabel('Giá thực tế (tỷ VND)')
plt.ylabel('Giá dự đoán (tỷ VND)')
plt.title('Random Forest: Dự đoán vs Thực tế', fontsize=14)
plt.grid(alpha=0.3)
plt.show()

## 15. Experiment 1: Model Comparison

In [ ]:
# So sánh mô hình - Biểu đồ
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# MAE
df_results[['MAE']].plot(kind='bar', ax=axes[0], color='skyblue')
axes[0].set_title('So sánh MAE các mô hình', fontsize=14)
axes[0].set_ylabel('MAE')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=45)

# R²
df_results[['R2']].plot(kind='bar', ax=axes[1], color='salmon')
axes[1].set_title('So sánh R² các mô hình', fontsize=14)
axes[1].set_ylabel('R²')
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=45)
axes[1].set_ylim(-0.1, 0.5)

plt.tight_layout()
plt.show()

## 16. Experiment 2: Hyperparameter Investigation

### 16.1 Decision Tree: Thay đổi max_depth

In [ ]:
print("="*60)
print("THÍ NGHIỆM 2A: ĐIỀU TRA SIÊU THAM SỐ DECISION TREE")
print("="*60)

depths = [3, 5, 10, None]
dt_results = {}

for depth in depths:
    model = DecisionTreeRegressor(random_state=42, max_depth=depth)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    dt_results[str(depth)] = {
        'MAE': mean_absolute_error(y_test, y_pred),
        'R2': r2_score(y_test, y_pred)
    }

dt_df = pd.DataFrame(dt_results).T
display(dt_df.round(4))

# Biểu đồ
fig, ax1 = plt.subplots(figsize=(8, 5))
color = 'tab:red'
ax1.set_xlabel('max_depth')
ax1.set_ylabel('MAE', color=color)
ax1.plot(dt_df.index, dt_df['MAE'], marker='o', color=color)
ax1.tick_params(axis='y', labelcolor=color)

ax2 = ax1.twinx()
color = 'tab:blue'
ax2.set_ylabel('R²', color=color)
ax2.plot(dt_df.index, dt_df['R2'], marker='s', color=color)
ax2.tick_params(axis='y', labelcolor=color)

plt.title('Decision Tree: max_depth vs MAE và R²', fontsize=14)
plt.grid(alpha=0.3)
plt.show()

### 16.2 Random Forest: Thay đổi n_estimators

In [ ]:
print("="*60)
print("THÍ NGHIỆM 2B: ĐIỀU TRA SIÊU THAM SỐ RANDOM FOREST")
print("="*60)

n_estimators_list = [50, 100, 200, 300]
rf_results = {}

for n in n_estimators_list:
    model = RandomForestRegressor(random_state=42, n_estimators=n)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    rf_results[n] = {
        'MAE': mean_absolute_error(y_test, y_pred),
        'R2': r2_score(y_test, y_pred)
    }

rf_df = pd.DataFrame(rf_results).T
display(rf_df.round(4))

# Biểu đồ
plt.figure(figsize=(8, 5))
plt.plot(rf_df.index, rf_df['MAE'], marker='o', label='MAE')
plt.plot(rf_df.index, rf_df['R2'], marker='s', label='R²')
plt.xlabel('Số lượng cây (n_estimators)')
plt.ylabel('Điểm số')
plt.title('Random Forest: n_estimators vs MAE và R²', fontsize=14)
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 17. Experiment 3: Representation / Feature Investigation

In [ ]:
print("="*60)
print("THÍ NGHIỆM 3: ẢNH HƯỞNG CỦA CHUẨN HÓA (SVR)")
print("="*60)

In [ ]:
# Không chuẩn hóa (sử dụng dữ liệu gốc)
X_train_raw = X_train.copy()
X_test_raw = X_test.copy()

# Thay thế các giá trị đã chuẩn hóa bằng dữ liệu gốc để so sánh
# Lấy lại dữ liệu gốc từ preprocessor
X_processed_raw = preprocessor.fit_transform(X_housing)
X_processed_raw = pd.DataFrame(X_processed_raw, columns=['Area', 'Floors', 'Bedrooms', 'Bathrooms', 
                                                          'Legal_encoded', 'Furniture_encoded'])

X_train_raw, X_test_raw, y_train_raw, y_test_raw = train_test_split(
    X_processed_raw, y_housing, test_size=0.20, random_state=42
)

# SVR không chuẩn hóa
svr_no_scaling = SVR(kernel='rbf', C=1.0, epsilon=0.2)
svr_no_scaling.fit(X_train_raw, y_train_raw)
y_pred_no_scale = svr_no_scaling.predict(X_test_raw)

mae_no_scale = mean_absolute_error(y_test_raw, y_pred_no_scale)
r2_no_scale = r2_score(y_test_raw, y_pred_no_scale)

# SVR có chuẩn hóa (đã có từ phần trước)
mae_with_scale = mae_svr
r2_with_scale = r2_svr

# So sánh
comparison_df = pd.DataFrame({
    'Phương pháp': ['Không chuẩn hóa', 'Có chuẩn hóa'],
    'MAE': [mae_no_scale, mae_with_scale],
    'R²': [r2_no_scale, r2_with_scale]
})
display(comparison_df)

# Biểu đồ
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
comparison_df.set_index('Phương pháp')[['MAE']].plot(kind='bar', ax=axes[0], color='skyblue')
axes[0].set_title('MAE: Không chuẩn hóa vs Có chuẩn hóa', fontsize=14)
axes[0].set_ylabel('MAE')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=0)

comparison_df.set_index('Phương pháp')[['R²']].plot(kind='bar', ax=axes[1], color='salmon')
axes[1].set_title('R²: Không chuẩn hóa vs Có chuẩn hóa', fontsize=14)
axes[1].set_ylabel('R²')
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=0)

plt.tight_layout()
plt.show()

## 18. Final Model

In [ ]:
# Chọn mô hình tốt nhất dựa trên kết quả thí nghiệm
print("="*60)
print("MÔ HÌNH CUỐI CÙNG: GRADIENT BOOSTING REGRESSOR")
print("="*60)

# Huấn luyện lại với toàn bộ dữ liệu train
final_model = GradientBoostingRegressor(
    random_state=42, 
    n_estimators=100, 
    learning_rate=0.1
)
final_model.fit(X_train, y_train)

# Đánh giá trên test
y_pred_final = final_model.predict(X_test)

mae_final = mean_absolute_error(y_test, y_pred_final)
mse_final = mean_squared_error(y_test, y_pred_final)
rmse_final = np.sqrt(mse_final)
r2_final = r2_score(y_test, y_pred_final)

print(f"MAE: {mae_final:.4f}")
print(f"MSE: {mse_final:.4f}")
print(f"RMSE: {rmse_final:.4f}")
print(f"R²: {r2_final:.4f}")

# Scatter plot: Dự đoán vs Thực tế
plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_pred_final, alpha=0.5, color='green')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
plt.xlabel('Giá thực tế (tỷ VND)')
plt.ylabel('Giá dự đoán (tỷ VND)')
plt.title('Final Model (Gradient Boosting): Dự đoán vs Thực tế', fontsize=14)
plt.grid(alpha=0.3)
plt.show()

## 19. Application

In [ ]:
def predict_house_price(area, floors, bedrooms, bathrooms, legal_status, furniture_state):
    """
    Dự đoán giá nhà sử dụng pipeline hoàn chỉnh.
    """
    # 1. Tạo DataFrame đầu vào
    input_data = pd.DataFrame({
        'Area': [area],
        'Floors': [floors],
        'Bedrooms': [bedrooms],
        'Bathrooms': [bathrooms],
        'Legal status': [legal_status],
        'Furniture state': [furniture_state]
    })
    
    # 2. Áp dụng toàn bộ pipeline tiền xử lý
    processed_features = preprocessor.transform(input_data)
    
    # 3. Dự đoán
    predicted_price = final_model.predict(processed_features)[0]
    
    return predicted_price

## 20. System Demonstration

In [ ]:
print("="*60)
print("DEMO HỆ THỐNG DỰ ĐOÁN GIÁ NHÀ")
print("="*60)

# Test cases
test_cases = [
    (84, 4, 4, 5, "Have certificate", "Full"),     # Case 1
    (60, 5, 4, 4, "Sale contract", "Basic"),       # Case 2
    (120, 3, 4, 5, "Have certificate", "Full")     # Case 3
]

for i, case in enumerate(test_cases, 1):
    price = predict_house_price(*case)
    print(f"\nCase {i}:")
    print(f"  Area: {case[0]}m², Floors: {case[1]}, Bedrooms: {case[2]}, Bathrooms: {case[3]}")
    print(f"  Legal: {case[4]}, Furniture: {case[5]}")
    print(f"  → Giá dự đoán: {price:.2f} tỷ VND")

In [ ]:
import joblib
import os

# Tạo thư mục models nếu chưa có
if not os.path.exists('models'):
    os.makedirs('models')

# Lưu model Gradient Boosting và scaler
joblib.dump(final_model, 'models/housing_model_gradient_boosting.pkl')
joblib.dump(housing_scaler, 'models/housing_scaler.pkl')

# Lưu cả preprocessor nếu cần
joblib.dump(preprocessor, 'models/housing_preprocessor.pkl')

print("Đã lưu model thành công vào thư mục 'models/'")
print("- housing_model_gradient_boosting.pkl")
print("- housing_scaler.pkl")
print("- housing_preprocessor.pkl")

## 21. Reflection

**1. Hệ thống nhận thông tin gì?**
- Hệ thống nhận các đặc điểm bất động sản: Area, Floors, Bedrooms, Bathrooms, Legal status, Furniture state.

**2. Biểu diễn nội bộ là gì?**
- Biểu diễn dưới dạng vector số x ∈ R^6 sau khi mã hóa và chuẩn hóa.

**3. Mô hình học gì từ các ví dụ?**
- Mô hình học mối quan hệ giữa vector đặc trưng và giá bán Price từ dữ liệu huấn luyện.

**4. Dự đoán/Quyết định gì?**
- Dự đoán giá trị số (Price) cho bất động sản mới.

**5. Tại sao có thể xử lý đầu vào chưa thấy?**
- Nhờ khả năng khái quát hóa của mô hình đã học từ dữ liệu huấn luyện.

**6. Phần nào có thể gọi là "thông minh"?**
- Phần mô hình tự động học và dự đoán là "thông minh".

**7. Hạn chế nào ngăn nó trở thành hệ thống thông minh mạnh mẽ hơn?**
- Thiếu thông tin vị trí chi tiết, nhiều giá trị thiếu, dữ liệu còn nhiễu.

**Phân biệt trained model và complete intelligent system:**
- **Trained model:** Chỉ là hàm toán học Random Forest.
- **Complete intelligent system:** Bao gồm input handling, preprocessing, model, output generation, và ứng dụng demo.

## 22. Conclusion

**Kết luận:**

1. Xây dựng thành công hệ thống dự đoán giá nhà với 5 mô hình học máy.
2. Random Forest là mô hình tốt nhất (R²=0.371, MAE=1.257 tỷ VND).
3. Gradient Boosting cũng cho hiệu suất cao (R²=0.330).
4. Chuẩn hóa dữ liệu cải thiện đáng kể SVR (R² từ -0.064 lên 0.150).
5. Cần bổ sung thêm thông tin về vị trí để cải thiện độ chính xác.